In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [ ]:
# Install lifelines silently
!pip install lifelines -q
!pip install lifelines
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter

warnings.filterwarnings('ignore')

print("📂 Loading datasets...")
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv', index_col='id')

plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# Map target to numeric for survival analysis
churn_numeric = train['Churn'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})

kmf = KaplanMeierFitter()
kmf.fit(durations=train['tenure'], event_observed=churn_numeric, label='Overall Customer Base')
ax = kmf.plot_survival_function(color='navy', linewidth=2.5)

if 'InternetService' in train.columns:
    kmf_fiber = KaplanMeierFitter()
    fiber_mask = train['InternetService'] == 'Fiber optic'
    kmf_fiber.fit(durations=train[fiber_mask]['tenure'], 
                  event_observed=churn_numeric[fiber_mask], 
                  label='Fiber Optic')
    kmf_fiber.plot_survival_function(ax=ax, color='crimson', linewidth=2, linestyle='--')

plt.title('Kaplan-Meier Estimate: Customer Retention Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Tenure (Months)', fontsize=12)
plt.ylabel('Probability of Retention (Survival)', fontsize=12)
plt.axvspan(0, 12, color='gold', alpha=0.15, label='High-Risk First Year')
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Standard Library Imports
import os
from subprocess import check_output
import warnings

# Third-Party Library Imports
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from scipy.stats import rankdata
from xgboost import XGBClassifier

# Scikit-Learn Imports (Specific Sub-modules)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import NearestNeighbors

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# Required: !pip install category_encoders -q
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

X = train.drop('Churn', axis=1)
y = train['Churn'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})

# --- PHASE 2: CUSTOM ENGINEERING ---
class ChurnBehavioralEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X_copy = X.copy()
        if 'tenure' in X_copy.columns:
            X_copy['early_tenure_risk'] = (X_copy['tenure'] <= 6).astype(int)
            X_copy['loyal_customer'] = (X_copy['tenure'] > 60).astype(int)
        if 'Partner' in X_copy.columns and 'Dependents' in X_copy.columns:
            X_copy['isolated_customer'] = ((X_copy['Partner'] == 'No') & (X_copy['Dependents'] == 'No')).astype(int)
        if 'TotalCharges' in X_copy.columns and 'MonthlyCharges' in X_copy.columns and 'tenure' in X_copy.columns:
            X_copy['TotalCharges'] = pd.to_numeric(X_copy['TotalCharges'], errors='coerce').fillna(0)
            X_copy['expected_total'] = X_copy['tenure'] * X_copy['MonthlyCharges']
            X_copy['cost_divergence'] = X_copy['TotalCharges'] - X_copy['expected_total']
        return X_copy

# --- PHASE 3: TARGET ENCODING & MODEL SETUP ---
preprocessor_advanced = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), make_column_selector(dtype_include=np.number)),
        ('cat_target', TargetEncoder(smoothing=10), make_column_selector(dtype_include=['object', 'category']))
    ], remainder='passthrough'
)

models = {
    'LGBM': LGBMClassifier(n_estimators=1500, learning_rate=0.015, max_depth=6, subsample=0.8, random_state=42, verbose=-1, n_jobs=-1),
    'CAT': CatBoostClassifier(iterations=1500, learning_rate=0.015, depth=6, l2_leaf_reg=3, verbose=0, random_seed=42, thread_count=-1),
    'XGB': XGBClassifier(n_estimators=1500, learning_rate=0.015, max_depth=5, subsample=0.8, tree_method='hist', random_state=42, verbosity=0, n_jobs=-1)
}
print("🛠️ Engineering and Models Initialized.")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

# --- PHASE 4: 10-FOLD LOOP & OPTIMIZATION ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_df = pd.DataFrame(index=X.index, columns=models.keys())
test_preds = pd.DataFrame(0.0, index=test.index, columns=models.keys())

print("🚀 Initiating 10-Fold Training Loop...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Apply Custom Engineering
    engineer = ChurnBehavioralEngineer()
    X_tr_eng = engineer.fit_transform(X_tr)
    X_val_eng = engineer.transform(X_val)
    test_eng = engineer.transform(test)
    
    # Apply Target Encoding strictly on training split to prevent leakage
    X_tr_proc = preprocessor_advanced.fit_transform(X_tr_eng, y_tr)
    X_val_proc = preprocessor_advanced.transform(X_val_eng)
    test_proc = preprocessor_advanced.transform(test_eng)
    
    print(f"   ↳ Fold {fold+1:02d} | Training independent models...")
    for name, model in models.items():
        model.fit(X_tr_proc, y_tr)
        oof_df.loc[val_idx, name] = model.predict_proba(X_val_proc)[:, 1]
        test_preds[name] += model.predict_proba(test_proc)[:, 1] / skf.n_splits

# Nelder-Mead Custom AUC Optimizer
def roc_auc_loss_function(weights, predictions, y_true):
    weights = weights / np.sum(weights) 
    blended_pred = np.zeros(len(y_true))
    for i in range(len(weights)):
        blended_pred += weights[i] * predictions.iloc[:, i].astype(float)
    return -roc_auc_score(y_true, blended_pred)

print("\n🏔️ Nelder-Mead Optimizer finding maximum AUC blend...")
initial_weights = [1.0 / len(models)] * len(models)
bounds = [(0.0, 1.0)] * len(models)

result = minimize(
    roc_auc_loss_function, 
    initial_weights, 
    args=(oof_df, y), 
    method='Nelder-Mead', 
    bounds=bounds,
    options={'maxiter': 1000}
)

optimal_weights = result.x / np.sum(result.x)
print(f"✅ Optimization Complete! Maximum Achieved AUC: {-result.fun:.6f}")
print(f"⚖️ Perfect Weights: LGBM: {optimal_weights[0]:.3f} | CAT: {optimal_weights[1]:.3f} | XGB: {optimal_weights[2]:.3f}")

# --- FINAL SUBMISSION ---
final_test_predictions = np.zeros(len(test))
for i, name in enumerate(models.keys()):
    final_test_predictions += optimal_weights[i] * test_preds[name]

submission = pd.DataFrame({'id': test.index, 'Churn': final_test_predictions})
submission.to_csv('submission_optimized.csv', index=False)
print("💾 submission_optimized.csv compiled using direct metric optimization.")